# [TITLE]

> Skeleton. Every bracketed placeholder is meant to be replaced.
> Formulas are referenced by name and source, not written out yet:
> decide the narration first, typeset the LaTeX last.

**One-line thesis:** [TODO]

**Reader assumed to know:** [TODO: linear algebra? attention? nothing?]

**Reader should leave knowing:** [TODO]

---

## How to use this skeleton

| Section field | Meaning |
|---|---|
| Bahas apa | scope of the section, one or two sentences |
| Rumus | which formula appears here, by name and source, LaTeX filled in later |
| Kode | what the code cell does, and where it comes from |
| Visual | which GIF from `assets/`, or a note if it does not exist yet |
| Note | open questions, things to decide, things to add |


In [ ]:
# Setup. Imports only, no logic.
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from IPython.display import Image, display

ASSETS = os.path.join("..", "assets")

# [TODO: decide whether to import from rope.py here or per-section]


---

# 1. [SECTION TITLE: the problem]

**Bahas apa**
[TODO: why attention is position-blind. The "kucing makan ikan (in mandarin)" vs
"ikan makan kucing (in mandarin)" observation.]

**Rumus**
- [FORMULA A] the plain attention score, dot product form. Source: Attention Is All You Need, Eq. [?]
- [FORMULA B] the relative-position constraint that RoPE must satisfy. Source: RoFormer Eq. 11

**Kode**
[TODO: is code needed here at all, or is this pure narration?
Candidate: a 4-line demo showing two different word orders producing
identical scores without position encoding.]

**Visual**
[TODO: none yet. Candidate: a small before/after showing identical
scores for reordered tokens. Decide if worth building. and score_matrix_animation.gif]

**Note**
[TODO: this section sets up the tension. Keep it short; the payoff is
section 2.]


In [ ]:
# [TODO: section 1 code]


---

# 2. [SECTION TITLE: rotation as the answer]

**Bahas apa**
[TODO: the dot product already depends only on the angle between two
vectors. Rotating each vector by its own position shifts that angle by
exactly the position difference. Absolute encoding, relative effect.]

**Rumus**
- [FORMULA C] geometric dot product, `a . b = |a||b| cos(angle)`. Source: standard
- [FORMULA D] angle after independent rotation. Source: derived in narration
- [FORMULA E] the 2D solution in complex form. Source: RoFormer Eq. 12
- [FORMULA F] orthogonality of R, and why norm is preserved. Source: standard

**Kode**
- `rope.rotate_2d` shown via `inspect.getsource`
- `test_invariance.py` run via subprocess to print live PASS/FAIL

**Visual**
- `assets/rope_rotation.gif` (discrete jumps + stored (x,y) table)

**Note**
[TODO: this is the core of the whole notebook. Consider whether the
"clock hands" framing belongs here or in section 4.]
[TODO: the norm-preservation point is what makes section 7's bug silent.
Plant the seed here explicitly so section 7 can pay it off.]


In [ ]:
# [TODO: inspect.getsource(rotate_2d)]


In [ ]:
# [TODO: subprocess run of test_invariance.py]


In [ ]:
# [TODO: display rope_rotation.gif]


---

# 3. [SECTION TITLE: the stronger claim]

**Bahas apa**
[TODO: "shift both together" is the weak version, true of any rigid
transform. The real claim: any two position pairs with the same
difference produce the same angle, even when the pairs are otherwise
unrelated.]

**Rumus**
- [reuse FORMULA D, stated in its stronger form]
- [FORMULA G] `R_m^T R_n = R_{n-m}`, the composition property. Source: RoFormer Eq. 16

**Kode**
[TODO: probably none. The test in section 2 already covers this via
`equal_difference_equal_score`. Decide: move that test's output here
instead of section 2?]

**Visual**
- `assets/equal_distance_animation.gif` (5 unrelated pairs, same delta)
- `assets/translation_invariance_animation.gif` (the weaker framing, for contrast)

**Note**
[TODO: order matters here. Show the weak version first then the strong
one, or lead with the strong one? Currently the GIFs exist for both.]


In [ ]:
# [TODO: display equal_distance_animation.gif]


In [ ]:
# [TODO: display translation_invariance_animation.gif]


---

# 4. [SECTION TITLE: why a spectrum of frequencies]

**Bahas apa**
[TODO: generalizing 2D to d dimensions splits into d/2 rotation planes.
Why not one frequency: cosine is periodic, distant positions alias onto
the same angle. Clock hands analogy.]

**Rumus**
- [FORMULA H] the block-diagonal rotation matrix R. Source: RoFormer Eq. 15
- [FORMULA I] the frequency spectrum `theta_i = base^(-2i/dim)`. Source: RoFormer Eq. 15
- [FORMULA J] efficient elementwise realization, avoiding the sparse matmul. Source: RoFormer Eq. 34

**Kode**
- `rope.inverse_frequencies` shown via `inspect.getsource`
- [TODO: also show `apply_rope_adjacent`? It demonstrates the strided
  implementation of FORMULA J. Decide.]

**Visual**
- `assets/frequency_spectrum_animation.gif` (16 planes, marker sweeping m)

**Note**
[TODO: the GIF currently uses dim=32 -> 16 planes -> 4x4 grid. Real
Qwen3-0.6B head_dim=128 -> 64 planes. Mention this so the reader does
not think 16 is canonical.]
[TODO: do not assert base=10000 universally. Say to read `rope_theta`
from the model config.]
[TODO: the decay property (section 8) falls out of this same choice of
spectrum. Foreshadow or leave it as a surprise?]


In [ ]:
# [TODO: inspect.getsource(inverse_frequencies)]


In [ ]:
# [TODO: display frequency_spectrum_animation.gif]


---

# 5. [SECTION TITLE: convention mismatch]

**Bahas apa**
[TODO: paper pairs adjacent dims (2i, 2i+1). HuggingFace pairs split
halves (i, i+dim/2) via rotate_half. Both valid, equivalent under a
permutation, silently wrong if mixed without it.]

**Rumus**
- [FORMULA K] the permutation mapping adjacent layout to split-half layout. Source: derived, see `rope_implementation_docs.md`
- [reference, not a formula] HF `rotate_half` and `apply_rotary_pos_emb` source snippet

**Kode**
- `test_conventions.py` run via subprocess
- [TODO: show `adjacent_to_half_permutation` source? It is the least
  obvious function in `rope.py`.]

**Visual**
[TODO: none exists. Candidate: a small diagram showing which dim index
pairs with which under each convention. Would be a static SVG/PNG, not
an animation. Worth building? Or is the table in the docs enough?]

**Note**
[TODO: this is the section most likely to be useful to someone else
debugging their own implementation. Consider making it standalone
readable.]
[TODO: mention the first-draft permutation bug from
`rope_implementation_docs.md` as a design note, or leave it in the docs
only?]


In [ ]:
# [TODO: subprocess run of test_conventions.py]


---

# 6. [SECTION TITLE: the score matrix]

**Bahas apa**
[TODO: what Q . K^T actually is, filled causally. Query t only attends
to keys 0..t.]

**Rumus**
- [reuse FORMULA A] the score as a dot product
- [FORMULA L] softmax over scores. Source: standard. [TODO: is softmax needed here or out of scope?]

**Kode**
[TODO: none currently. The GIF computes its own vectors. Decide whether
to show the example Q/K vectors inline so the reader can check the
arithmetic by hand.]

**Visual**
- `assets/score_matrix_animation.gif` (causal fill, one dot product per frame)

**Note**
[TODO: current GIF shows scores WITHOUT RoPE, on purpose, matching the
"before position is injected" framing. Decide: add a side-by-side with
RoPE applied, showing scores change? That was flagged as a natural
extension but not built.]
[TODO: placement. This section is arguably prerequisite to section 1,
not a follow-on from section 5. Reconsider ordering.]


In [ ]:
# [TODO: display score_matrix_animation.gif]


---

# 7. [SECTION TITLE: the silent bug]

**Bahas apa**
[TODO: cache truncation. Cached keys keep the rotation from their
original absolute positions; slicing a tensor does not undo a rotation
already applied. If the next query's position comes from the truncated
cache length, query and keys disagree about the frame, and part of the
relative distance range goes negative.]

**Rumus**
- [FORMULA M] the resulting relative distance range, `[-prefix, suffix - prefix]`. Source: derived
- [callback to FORMULA F] orthogonality is why this never crashes

**Kode**
- `test_position_mismatch.py` run via subprocess
- [TODO: show the HF snippet where position_ids is inferred from
  `past_key_values.get_seq_length()`? It is the literal source of the
  bug. Decide if quoting framework internals belongs in this notebook.]

**Visual**
- `assets/attention_mismatch_animation.gif` (two heatmaps, correct vs mismatched)

**Note**
[TODO: the heatmap color difference is subtle (max ~0.04). Flagged
earlier as a candidate for a third panel showing amplified difference.
Build it, or state the numbers in prose instead?]
[TODO: scope discipline. The synthetic test proves divergence exists at
every length, but NOT that it worsens for short sequences. Do not claim
the trend here. See section 9.]


In [ ]:
# [TODO: subprocess run of test_position_mismatch.py]


In [ ]:
# [TODO: display attention_mismatch_animation.gif]


---

# 8. [SECTION TITLE: long-term decay]

**Bahas apa**
[TODO: the rotary dot product's upper bound shrinks as relative distance
grows. Emergent from the frequency spectrum chosen in section 4, not a
separately designed feature.]

**Rumus**
- [FORMULA N] the Abel-transformed sum and its bound. Source: RoFormer Eq. 35-37
- [FORMULA O] the quantity actually plotted, `mean|S_j|` over partial sums. Source: RoFormer Eq. 37

**Kode**
[TODO: none currently. The GIF computes the bound itself. Decide whether
to show `relative_upper_bound` inline; it is short and the choice of
quantity is the whole point of this section.]

**Visual**
- `assets/long_term_decay_animation.gif` (bound vs distance, two dims, fill_between)

**Note**
[IMPORTANT] The first draft plotted the mean `|q . k|` over random
vectors and showed almost no decay (1.08x). The correct quantity is the
deterministic upper bound. Decide whether this correction is worth
narrating in the notebook or belongs only in
`rope_implementation_docs.md`.
[TODO: connect back to section 4 explicitly. One spectrum choice, two
consequences: no aliasing, and decay.]


In [ ]:
# [TODO: display long_term_decay_animation.gif]


---

# 9. [SECTION TITLE: scope and limits]

**Bahas apa**
[TODO: what this repo verifies vs what it cannot. Properties of the
encoding itself are in scope. How a positional error affects a specific
trained model's output depends on that model's weights and training
distribution, and is out of scope.]

**Rumus**
[none]

**Kode**
[none]

**Visual**
[none]

**Note**
[TODO: this section is the honesty guard. Without it the notebook
overclaims. Keep it, and keep it before any conclusion.]
[TODO: point to `rope_implementation_docs.md` for per-function
derivations and the design notes.]
[TODO: reference. Su et al., RoFormer, arXiv:2104.09864.]


---

# Backlog

Things flagged during construction, not yet decided:

| Item | Section | Status |
|---|---|---|
| [TODO] Side-by-side score matrix, with vs without RoPE | 6 | not built |
| [TODO] Third heatmap panel showing amplified difference | 7 | not built |
| [TODO] Static diagram of the two pairing conventions | 5 | not built |
| [TODO] Decide if HF framework snippets belong inline | 5, 7 | undecided |
| [TODO] Reorder: does section 6 belong before section 1? | 1, 6 | undecided |
| [TODO] Narrate the decay-quantity correction, or keep in docs? | 8 | undecided |
| [TODO] [add your own] | | |
